# Loan Account Cleaning
This notebook processes `Raw_Loan_Account.csv` (499 rows) to create a cleansed layer for Power BI, as part of an end-to-end ETL pipeline.
- **Staging**: Inspect data, note missing values, standardize formats, remove empty columns.
- **Cleansed**: Create `IsActive` and `LoanStatus`, evaluate `ChannelID`, save to `cleansed_loan_accounts.csv`.


In [2]:
import pandas as pd
# Load raw data with semicolon delimiter
df = pd.read_csv('Raw_Loan_Account.csv', delimiter=';', dtype_backend='numpy_nullable')

## Staging: Data Inspection
Inspect data types, missing values, duplicates, and `CancelledDate` distribution.

In [3]:
# Display first 5 rows
print("First 5 rows of raw data:")
print(df.head())

# Display data types and non-null counts
print("\nData info:")
print(df.info())

# Display missing values per column
print("\nNumber of NaN values per column:")
print(df.isna().sum())

First 5 rows of raw data:
   LoanAccountId  SourceId AccountNumber  IBAN  BBAN  AccountCurrencyId  \
0              1         8    1003250055  <NA>  <NA>                 49   
1              2         8    1004620017  <NA>  <NA>                 49   
2              3         8    1005760051  <NA>  <NA>                 49   
3              4         8    1005760051  <NA>  <NA>                 49   
4              5         8    1012630057  <NA>  <NA>                 49   

  AccountCurrency  OrganizationId OrganizationName  ChannelID  ...  \
0             EUR              22   Nordic Finland         29  ...   
1             EUR              40   Nordic Finland         28  ...   
2             EUR              22   Nordic Finland       <NA>  ...   
3             EUR              22   Nordic Finland       <NA>  ...   
4             EUR              22   Nordic Finland         27  ...   

    ValueDate  MaturityDate ProductId                   Product InvoiceDay  \
0  2019-10-12          <

In [4]:
# Check for duplicate AccountNumber (noted rows with duplicates)
account_counts = df['AccountNumber'].value_counts()
duplicate_accounts = account_counts[account_counts > 1]
print(f"\nNumber of duplicated AccountNumbers: {len(duplicate_accounts)}")
if not duplicate_accounts.empty:
    print("Duplicated AccountNumbers:")
    print(duplicate_accounts)



Number of duplicated AccountNumbers: 118
Duplicated AccountNumbers:
AccountNumber
5,11E+11      150
1027840097      2
1085590014      2
1087480016      2
1079380018      2
             ... 
1130900010      2
1129940019      2
1130510017      2
1133050094      2
1115600056      2
Name: count, Length: 118, dtype: Int64


5,11E+11            (150) - what is this?

## Staging: Remove Empty Columns
Remove columns with no data (`IBAN`, `BBAN`, `MaturityDate`, `RepaymentRate`).

In [5]:
# Define columns to drop (verified as NULL in Excel)
columns_to_drop = ['IBAN', 'BBAN', 'MaturityDate', 'RepaymentRate']

# Drop columns, ignore errors if columns don't exist
df = df.drop(columns=columns_to_drop, errors='ignore')

# Verify remaining columns
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['LoanAccountId', 'SourceId', 'AccountNumber', 'AccountCurrencyId', 'AccountCurrency', 'OrganizationId', 'OrganizationName', 'ChannelID', 'BrokerId', 'OpenDateId', 'OpenDate', 'CancelledDateId', 'CancelledDate', 'ValueDate', 'ProductId', 'Product', 'InvoiceDay', 'CurrentInstallmentAmount', 'CurrentInvoiceFee', 'NextInvoiceDate', 'CalculatedMaturityDate']


Data Inspection
Upon loading the `Raw_Loan_Account.csv` file, the columns `IBAN`, `BBAN`, `MaturityDate`, and `RepaymentRate` were found to contain only `NaN` values in the Pandas DataFrame. To confirm the absence of data, the CSV file was manually inspected in Excel, where all values in these columns were verified to be `NULL`. As these columns contain no usable information, they will be removed from the cleaned dataset.

In [6]:
# List of columns with no analytical value to be removed
columns_to_drop = [
    'IBAN', 'BBAN', 'MaturityDate', 'RepaymentRate'
]

# Drop the columns from the DataFrame
df = df.drop(columns=columns_to_drop, errors='ignore')

# Verify the columns have been removed
print("Remaining columns:", df.columns.tolist())

Remaining columns: ['LoanAccountId', 'SourceId', 'AccountNumber', 'AccountCurrencyId', 'AccountCurrency', 'OrganizationId', 'OrganizationName', 'ChannelID', 'BrokerId', 'OpenDateId', 'OpenDate', 'CancelledDateId', 'CancelledDate', 'ValueDate', 'ProductId', 'Product', 'InvoiceDay', 'CurrentInstallmentAmount', 'CurrentInvoiceFee', 'NextInvoiceDate', 'CalculatedMaturityDate']


## Staging: Standardize Formats
Convert date columns and numeric fields to consistent formats.

In [7]:
# Define date columns for conversion
date_columns = ['OpenDate', 'CancelledDate', 'ValueDate', 'NextInvoiceDate', 'CalculatedMaturityDate']

# Convert to datetime, handle errors
df[date_columns] = df[date_columns].apply(pd.to_datetime, errors='coerce')

# Convert CurrentInstallmentAmount to numeric, handle comma as decimal
df['CurrentInstallmentAmount'] = df['CurrentInstallmentAmount'].astype(str).str.replace(',', '.')
df['CurrentInstallmentAmount'] = pd.to_numeric(df['CurrentInstallmentAmount'], errors='coerce')

# Verify updated data types
print("\nUpdated data types:")
print(df.dtypes)


Updated data types:
LoanAccountId                        Int64
SourceId                             Int64
AccountNumber               string[python]
AccountCurrencyId                    Int64
AccountCurrency             string[python]
OrganizationId                       Int64
OrganizationName            string[python]
ChannelID                            Int64
BrokerId                             Int64
OpenDateId                           Int64
OpenDate                    datetime64[ns]
CancelledDateId                      Int64
CancelledDate               datetime64[ns]
ValueDate                   datetime64[ns]
ProductId                            Int64
Product                     string[python]
InvoiceDay                           Int64
CurrentInstallmentAmount           float64
CurrentInvoiceFee           string[python]
NextInvoiceDate             datetime64[ns]
CalculatedMaturityDate      datetime64[ns]
dtype: object


## Staging: Handle AccountNumber

- Scientific notation is converted to standard numeric format, but kept as a string for display purposes.
- Padding with zeros ensures consistent 12-character length.
- Ensure AccountNumber remains a string to prevent unintended aggregation in Power BI.

In [8]:
# Standardize AccountNumber: handle scientific notation, pad to 12 characters
df['AccountNumber'] = df['AccountNumber'].astype(str).apply(lambda x: str(int(float(x.replace(',', '')))) if 'E' in x else x)
df['AccountNumber'] = df['AccountNumber'].str.zfill(12)

# Cleansed Layer
Perform transformations to create a clean dataset for Power BI:
- Create `IsActive` and `LoanStatus` based on `CancelledDate`.
- Evaluate `ChannelID`and duplicate `AccountNumber` .
- Save clean data to `cleansed/cleansed_loan_accounts.csv`.

## Cleansed: Create IsActive and LoanStatus
Create columns to indicate loan status based on `CancelledDate` for reporting.

In [9]:
# Create IsActive: 1 if CancelledDate is NaN (active), 0 if CancelledDate exists (cancelled)
df['IsActive'] = df['CancelledDate'].isna().astype(int)

# Create LoanStatus for readability in Power BI
df['LoanStatus'] = df['IsActive'].map({1: 'Active', 0: 'Cancelled'})

# Verify distribution of IsActive
print("Distribution of IsActive (1 = active, 0 = cancelled):")
print(df['IsActive'].value_counts())


Distribution of IsActive (1 = active, 0 = cancelled):
IsActive
1    287
0    212
Name: count, dtype: int64


## Cleansed: Handle ChannelID and Duplicate Account Numbers

- Evaluate `ChannelID`
Assess whether `ChannelID` should be removed due to a high percentage of missing values, ensuring data integrity and relevance.
- Handle Duplicate `AccountNumber`
Review duplicate entries in `AccountNumber` to determine whether they indicate redundant records or necessary financial transactions.


In [10]:
# Check missing values in ChannelID
missing_ratio = df['ChannelID'].isnull().mean()
print(f"Missing values in ChannelID: {missing_ratio:.2%}")

# Check duplicate AccountNumbers
duplicate_count = df.duplicated(subset=['AccountNumber']).sum()
print(f"Duplicate AccountNumber count: {duplicate_count}")


Missing values in ChannelID: 76.95%
Duplicate AccountNumber count: 266


## Decision: Retaining `ChannelID` and Duplicate `AccountNumber`

- Although `ChannelID` has a high percentage of missing values, it is retained to avoid premature data loss. Keeping it allows for potential future analysis, external data integration, or pattern recognition that may still be valuable.
- Duplicate `AccountNumber` entries are preserved as their relevance depends on contextual information from other datasets. Removing them without verifying their relationships could lead to data inconsistencies, affecting the overall accuracy of the analysis.


In [11]:
print(df.describe)

<bound method NDFrame.describe of      LoanAccountId  SourceId   AccountNumber  AccountCurrencyId  \
0                1         8    001003250055                 49   
1                2         8    001004620017                 49   
2                3         8    001005760051                 49   
3                4         8    001005760051                 49   
4                5         8    001012630057                 49   
..             ...       ...             ...                ...   
494            495         8  51100000000000                 49   
495            496         8  51100000000000                 49   
496            497         8  51100000000000                 49   
497            498         8  51100000000000                 49   
498            499         8  51100000000000                 49   

    AccountCurrency  OrganizationId OrganizationName  ChannelID  BrokerId  \
0               EUR              22   Nordic Finland         29       173   
1      

## Cleansed: Save Clean Data
Save cleansed data for analytical layer and Power BI.

In [12]:
# Save cleansed data
df.to_csv('cleansed_loan_account.csv', index=False)
print("Saved cleansed data to cleansed_loan_account.csv.")

Saved cleansed data to cleansed_loan_account.csv.


In [13]:
print(df.columns)

Index(['LoanAccountId', 'SourceId', 'AccountNumber', 'AccountCurrencyId',
       'AccountCurrency', 'OrganizationId', 'OrganizationName', 'ChannelID',
       'BrokerId', 'OpenDateId', 'OpenDate', 'CancelledDateId',
       'CancelledDate', 'ValueDate', 'ProductId', 'Product', 'InvoiceDay',
       'CurrentInstallmentAmount', 'CurrentInvoiceFee', 'NextInvoiceDate',
       'CalculatedMaturityDate', 'IsActive', 'LoanStatus'],
      dtype='object')
